1. XML 파일 로드
2. XML 구조 탐색
3. 주요 태그 확인
4. annotation 구조 이해
5. JSON label 확인
6. extract_nodule_info() 구현
7. 테스트

In [ ]:
# =========================================================
# 데이터 경로 확인
# =========================================================
import os

PATH = "/home/hdo/lidc_project/data"
BASE_PATH = "/home/hdo/lidc_project/data/lidc-idri"

print(os.listdir(PATH))
print(os.listdir(BASE_PATH))

['lidc-idri']
['LIDC-XML-only', 'manifest-1600709154662', 'slices_png', 'lidc-idri-nodule-counts-6-23-2015.xlsx', 'slices', 'tcia-diagnosis-data-2012-04-20.xls', 'nodule_malignancy_scores.json']


In [ ]:
# =========================================================
# XML 파일 수집
# =========================================================
import glob

xml_files = glob.glob(
    os.path.join(BASE_PATH, "LIDC-XML-only/**/*.xml"),
    recursive=True
)

print("Number of XML files:", len(xml_files))
print("First XML:", xml_files[0])

Number of XML files: 1319
First XML: /home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/161-resubmitted-correction-3-9-12.xml


3. XML 하나 파싱

In [ ]:
# =========================================================
# PARSE ONE XML
# =========================================================
import xml.etree.ElementTree as ET

tree = ET.parse(xml_files[0])
root = tree.getroot()

print("ROOT TAG:")
print(root.tag)

ROOT TAG:
{http://www.nih.gov}LidcReadMessage


4. XML 구조 탐색

In [ ]:
# =========================================================
# XML 구조 탐색
# =========================================================

print("\n===== XML TAG STRUCTURE =====")

for i, elem in enumerate(root.iter()):

    print(elem.tag)

    if i > 30:
        break


===== XML TAG STRUCTURE =====
{http://www.nih.gov}LidcReadMessage
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}Version
{http://www.nih.gov}MessageId
{http://www.nih.gov}DateRequest
{http://www.nih.gov}TimeRequest
{http://www.nih.gov}RequestingSite
{http://www.nih.gov}ServicingSite
{http://www.nih.gov}TaskDescription
{http://www.nih.gov}CtImageFile
{http://www.nih.gov}SeriesInstanceUid
{http://www.nih.gov}StudyInstanceUID
{http://www.nih.gov}DateService
{http://www.nih.gov}TimeService
{http://www.nih.gov}ResponseDescription
{http://www.nih.gov}ResponseComments
{http://www.nih.gov}readingSession
{http://www.nih.gov}annotationVersion
{http://www.nih.gov}servicingRadiologistID
{http://www.nih.gov}unblindedReadNodule
{http://www.nih.gov}noduleID
{http://www.nih.gov}roi
{http://www.nih.gov}imageZposition
{http://www.nih.gov}imageSOP_UID
{http://www.nih.gov}inclusion
{http://www.nih.gov}edgeMap
{http://www.nih.gov}xCoord
{http://www.nih.gov}yCoord
{http://www.nih.gov}unblindedReadN

5. 주요 태그 존재 여부 확인

In [ ]:
# =========================================================
# 주요 태그 존재 여부 확인
# =========================================================

tags = set()

for elem in root.iter():
    tags.add(elem.tag)

print("\n===== TAG LIST =====")

for tag in sorted(tags):
    print(tag)


===== TAG LIST =====
{http://www.nih.gov}CtImageFile
{http://www.nih.gov}DateRequest
{http://www.nih.gov}DateService
{http://www.nih.gov}LidcReadMessage
{http://www.nih.gov}MessageId
{http://www.nih.gov}RequestingSite
{http://www.nih.gov}ResponseComments
{http://www.nih.gov}ResponseDescription
{http://www.nih.gov}ResponseHeader
{http://www.nih.gov}SeriesInstanceUid
{http://www.nih.gov}ServicingSite
{http://www.nih.gov}StudyInstanceUID
{http://www.nih.gov}TaskDescription
{http://www.nih.gov}TimeRequest
{http://www.nih.gov}TimeService
{http://www.nih.gov}Version
{http://www.nih.gov}annotationVersion
{http://www.nih.gov}calcification
{http://www.nih.gov}characteristics
{http://www.nih.gov}edgeMap
{http://www.nih.gov}imageSOP_UID
{http://www.nih.gov}imageZposition
{http://www.nih.gov}inclusion
{http://www.nih.gov}internalStructure
{http://www.nih.gov}lobulation
{http://www.nih.gov}locus
{http://www.nih.gov}malignancy
{http://www.nih.gov}margin
{http://www.nih.gov}noduleID
{http://www.nih.

In [26]:
# =========================================================
# SeriesInstanceUid 찾기
# =========================================================

print("\n===== SERIES INSTANCE UID =====")

for elem in root.iter():

    if "SeriesInstanceUid" in elem.tag:

        print("Series UID:", elem.text)


# =========================================================
# malignancy 찾기
# =========================================================

print("\n===== MALIGNANCY =====")

for elem in root.iter():

    if "malignancy" in elem.tag:

        print("malignancy:", elem.text)


# =========================================================
# imageZposition 찾기
# =========================================================

print("\n===== IMAGE Z POSITION =====")

z_count = 0

for elem in root.iter():

    if "imageZposition" in elem.tag:

        print("z:", elem.text)

        z_count += 1

        if z_count > 10:
            break


===== SERIES INSTANCE UID =====
Series UID: 1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327

===== MALIGNANCY =====
malignancy: 1
malignancy: 3

===== IMAGE Z POSITION =====
z: -253.0
z: -253.0
z: -241.0
z: -262.0
z: -187.0
z: -262.0
z: -271.0
z: -70.0
z: -76.0
z: -121.0
z: -181.0


In [ ]:
# =========================================================
# edgeMap 좌표 확인
# =========================================================

print("\n===== EDGE MAP COORDINATES =====")

edge_count = 0

for elem in root.iter():

    if "edgeMap" in elem.tag:

        x = None
        y = None

        for child in elem:

            if "xCoord" in child.tag:
                x = child.text

            if "yCoord" in child.tag:
                y = child.text

        print(f"(x={x}, y={y})")

        edge_count += 1

        if edge_count > 20:
            break


===== EDGE MAP COORDINATES =====
(x=49, y=310)
(x=202, y=396)
(x=185, y=392)
(x=71, y=209)
(x=398, y=252)
(x=270, y=285)
(x=65, y=254)
(x=60, y=198)
(x=185, y=393)
(x=49, y=316)
(x=50, y=315)
(x=51, y=314)
(x=51, y=313)
(x=51, y=312)
(x=51, y=311)
(x=51, y=310)
(x=51, y=309)
(x=51, y=308)
(x=51, y=307)
(x=50, y=306)
(x=49, y=305)


## annotation 구조 확인

In [30]:
# =========================================================
# readingSession 개수 확인
# =========================================================

print("\n===== READING SESSION COUNT =====")

session_count = 0

for elem in root.iter():

    if "readingSession" in elem.tag:

        session_count += 1

print("Number of reading sessions:", session_count)


# =========================================================
# unblindedReadNodule 개수 확인
# =========================================================

print("\n===== NODULE COUNT =====")

nodule_count = 0

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        nodule_count += 1

print("Number of nodules:", nodule_count)



===== READING SESSION COUNT =====
Number of reading sessions: 4

===== NODULE COUNT =====
Number of nodules: 21


In [ ]:
# =========================================================
# 결절 하나 상세 구조 출력
# =========================================================
print("\n===== FIRST NODULE DETAIL =====")

for elem in root.iter():

    if "unblindedReadNodule" in elem.tag:

        for child in elem:

            print("\nCHILD TAG:", child.tag)

            for sub in child.iter():

                print("   ", sub.tag, ":", sub.text)

        break


===== FIRST NODULE DETAIL =====

CHILD TAG: {http://www.nih.gov}noduleID
    {http://www.nih.gov}noduleID : 0

CHILD TAG: {http://www.nih.gov}roi
    {http://www.nih.gov}roi : 
    
    {http://www.nih.gov}imageZposition : -253.0
    {http://www.nih.gov}imageSOP_UID : 1.3.6.1.4.1.14519.5.2.1.6279.6001.150739457477763063347777523734
    {http://www.nih.gov}inclusion : TRUE
    {http://www.nih.gov}edgeMap : None
    {http://www.nih.gov}xCoord : 49
    {http://www.nih.gov}yCoord : 310


In [32]:
# 실제 UID 기준 데이터셋 개수 확인
uid_set = set()

for xml_path in xml_files:

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        for elem in root.iter():

            if "SeriesInstanceUid" in elem.tag:

                uid_set.add(elem.text)
                break

    except:
        continue

print("Unique CT Series:", len(uid_set))

Unique CT Series: 1018


In [34]:
# =========================================================
# 12. JSON LABEL CHECK
# =========================================================
import json

json_path = os.path.join(
    BASE_PATH,
    "nodule_malignancy_scores.json"
)

with open(json_path, "r") as f:
    data = json.load(f)

print(type(data))
print(list(data.keys())[:10])

<class 'dict'>
['LIDC-IDRI-0173', 'LIDC-IDRI-0253', 'LIDC-IDRI-0189', 'LIDC-IDRI-0193', 'LIDC-IDRI-0240', 'LIDC-IDRI-0251', 'LIDC-IDRI-0241', 'LIDC-IDRI-0131', 'LIDC-IDRI-0212', 'LIDC-IDRI-0122']


# 실제 데이터 추출 함수

In [35]:
# =========================================================
# 13. EXTRACT NODULE INFO FUNCTION
# =========================================================

def extract_nodule_info(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    series_uid = None

    # -----------------------------------------------------
    # Series UID
    # -----------------------------------------------------

    for elem in root.iter():

        if "SeriesInstanceUid" in elem.tag:

            series_uid = elem.text
            break

    nodules = []

    # -----------------------------------------------------
    # Nodule Parsing
    # -----------------------------------------------------

    for nodule in root.iter():

        if "unblindedReadNodule" not in nodule.tag:
            continue

        malignancy = None
        coords = []

        # characteristic 내부 탐색
        for sub in nodule.iter():

            # malignancy
            if "malignancy" in sub.tag:
                malignancy = sub.text

            # edgeMap
            if "edgeMap" in sub.tag:

                x = None
                y = None

                for child in sub:

                    if "xCoord" in child.tag:
                        x = child.text

                    if "yCoord" in child.tag:
                        y = child.text

                coords.append((x, y))

        nodules.append({
            "series_uid": series_uid,
            "malignancy": malignancy,
            "coords": coords
        })

    return nodules

In [36]:
# =========================================================
# 14. FUNCTION TEST
# =========================================================

sample = extract_nodule_info(xml_files[0])

print("Number of nodules:", len(sample))

print(sample[0])

Number of nodules: 21
{'series_uid': '1.3.6.1.4.1.14519.5.2.1.6279.6001.340202188094259402036602717327', 'malignancy': None, 'coords': [('49', '310')]}
